In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Config

In [18]:
import sys
from pathlib import Path

# Find the repo root by walking up until we hit src/A_Config.py -- see
# AA_Cam_pose_estimate_batches.ipynb for why this is more robust than Path.cwd().parent.
project_root = Path.cwd()
while not (project_root / "src" / "A_Config.py").exists():
    if project_root == project_root.parent:
        raise RuntimeError("couldn't find repo root (src/A_Config.py) above cwd")
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root /"src" / "Experiments" / "templates"))
print("project_root:", project_root)

from A_Config import set_case, case_dir
from Two2D.B_video_processing import video_fps
from AA_cam_pose_sweeps import reset_sweep_state, run_density_sweep
case_name = "02_Flat"
experiment = "explore"
set_case(case_name, experiment)  # must happen before any case_dir()/A_Config getter call below

source_dir   = case_dir() / "010_source"
videos = list(source_dir.glob("*.mp4"))
video_path = videos[0]
import cv2; total_frames = int(cv2.VideoCapture(str(video_path)).get(cv2.CAP_PROP_FRAME_COUNT))
print("total_frames",total_frames )
# Fixed video range -- only the frame DENSITY within this range varies across the sweep.
# Hardcoded per AI_coppertest3_vggt.ipynb's precedent (no reusable start/end-frame config exists yet).
start_3D_recon = 0
end_3D_recon = 5117


fps = video_fps(video_path)
print(f"video fps: {fps:.3f}")

# Target frame counts to sweep, ascending -- ascending order matters for the
# OOM-precaution logic in run_density_sweep (more frames -> more GPU memory).
CAM_POSES = [25, 50, 75, 100, 125, 150, 175, 200, 225, 250,275,300]

# Fixed run order (user-specified) -- cheapest/most-reliable techniques first.
TECHNIQUE_ORDER = ["CUT3R","CUT3R_Revisit", "lingbot-map", "VGGT-Omega", "MegaSaM", "VGGT"]

# Which techniques are actually enabled -- same defaults as AA_Cam_pose_estimate_batches.ipynb.
FLAGS = {
    "CUT3R":       False,
    "CUT3R_Revisit": False,
    "lingbot-map": False,
    "VGGT-Omega":  True,
    "MegaSaM":     False,
    "VGGT":        False,
}

# What the sweep keeps on disk afterwards vs deletes -- see AA_cam_pose_sweeps.py's
# _cleanup_technique_output for exactly what each toggle controls.
DATA_STORAGE = {
    "keep_inputs":     False,  # extracted frames per density (020_frames_for_cam_poses)
    "keep_recon_data": False,  # full heavy technique output (images/depth/point-clouds/caches)
    "keep_cam_poses":  True,   # lightweight <technique_output>/CAM_poses/poses.csv per density
    "full_pose":       True,  # rotation too, not just translation, in the retained CSV
    "render_bev":      True,   # one <technique_output>/BEV/bev.png per technique/density --
                                # every technique gets a Reconstruction (natively for VGGT-Omega,
                                # via F_transpose_to_recon_objects.py adapters for the rest), then
                                # P_projection_mapping.BEV_tile_render_PM renders from it. Cheap
                                # to keep even when keep_recon_data=False.
}

project_root: /cs/student/project_msc/2025/cgvi/dbarker
total_frames 5117
video fps: 30.000


## Reset sweep state

Deletes **only** this sweep's own output -- everything from running this sweep, nothing more, nothing less: every `poses_*` frame/reconstruction folder under the `experiment` tag defined above, plus the status/results CSVs. Nothing from `AA_Cam_pose_estimate_batches.ipynb` (different experiment name) or anything else in the case directory is touched.

Safe by default: with `CONFIRM_DELETE = False` this only prints what it *would* delete. Flip it to `True` and re-run the cell to actually delete.

In [19]:
CONFIRM_DELETE = False  # flip to True and re-run this cell to actually delete

reset_sweep_state(case_name, experiment, confirm=CONFIRM_DELETE)

confirm=False -- dry run only, nothing deleted. Would remove:
  rmtree /cs/student/project_msc/2025/cgvi/dbarker/Data/02_Flat/020_frames_for_cam_poses/explore
  rmtree /cs/student/project_msc/2025/cgvi/dbarker/Data/02_Flat/035_VGGT_O_output/explore


In [20]:
(end_3D_recon - start_3D_recon)

5117

## Part 2 -- Sweep (expensive, resumable, run once/rarely)

`run_density_sweep` (in `AA_cam_pose_sweeps.py`) gives each `cam_poses` value its own frames/output folders via a nested experiment tag (`f"{experiment}/poses_{cam_poses:04d}"` passed to `set_case`) -- this makes every `A_Config` getter (including `run_lingbot_map`'s, which has no path parameters at all and only reads the zero-arg getters) automatically density-scoped, with no per-function path-wrangling needed.

Safe to interrupt/kill and re-run from the top at any point.

In [21]:
run_density_sweep(
    case_name, experiment, video_path, start_3D_recon, end_3D_recon,
    CAM_POSES, TECHNIQUE_ORDER, FLAGS, DATA_STORAGE, project_root,
)

Video   : VID_20260519_103250983.mp4
          30.1 fps · 5117 frames · 170.1s
Sampling: every 206 frames (6.833333333333333s) → 25 candidates
Window  : ±7 frames  |  scoring at 25% resolution


Pass 1: scoring: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 5117/5117 [00:06<00:00, 736.06fr/s]


Selected: 25 frames  (22 shifted from sample point, 0 skipped below threshold)


Pass 3: writing: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 5117/5117 [00:03<00:00, 1561.12fr/s]


Log     : /cs/student/project_msc/2025/cgvi/dbarker/Data/02_Flat/020_frames_for_cam_poses/explore/poses_0025/selection_log.json
Sharpness (at 25% res): min 21 · mean 823 · max 2424
25 frames  →  /cs/student/project_msc/2025/cgvi/dbarker/Data/02_Flat/035_VGGT_O_output/explore/poses_0025
Saved: /cs/student/project_msc/2025/cgvi/dbarker/Data/02_Flat/035_VGGT_O_output/explore/poses_0025/predictions.npz
Saved: /cs/student/project_msc/2025/cgvi/dbarker/Data/02_Flat/035_VGGT_O_output/explore/poses_0025/scene.glb
[6, 200, 405, 619, 823, 1024, 1234, 1446, 1648, 1847, 2061, 2265, 2478, 2677, 2890, 3088, 3298, 3502, 3706, 3919, 4113, 4322, 4532, 4732, 4940]
tensor(2.8000, device='cuda:0')
tensor(2.6036, device='cuda:0')
tensor(2.5916, device='cuda:0')
tensor(2.7629, device='cuda:0')
tensor(0.8703, device='cuda:0')
tensor(0.8535, device='cuda:0')
tensor(0.8873, device='cuda:0')
tensor(0.8787, device='cuda:0')
tensor(2.0411, device='cuda:0')
tensor(2.0671, device='cuda:0')
tensor(2.0803, device='cu

...

Log     : /cs/student/project_msc/2025/cgvi/dbarker/Data/02_Flat/020_frames_for_cam_poses/explore/poses_0275/selection_log.json
Sharpness (at 25% res): min 2 · mean 926 · max 4656
[VGGT-Omega @ 275] already failed (skipped_precautionary), skipping
Video   : VID_20260519_103250983.mp4
          30.1 fps · 5117 frames · 170.1s
Sampling: every 18 frames (0.6s) → 285 candidates
Window  : ±7 frames  |  scoring at 25% resolution


Pass 1: scoring: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 5117/5117 [00:06<00:00, 794.29fr/s]


Selected: 285 frames  (239 shifted from sample point, 0 skipped below threshold)


Pass 3: writing: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 5117/5117 [00:04<00:00, 1250.09fr/s]


Log     : /cs/student/project_msc/2025/cgvi/dbarker/Data/02_Flat/020_frames_for_cam_poses/explore/poses_0300/selection_log.json
Sharpness (at 25% res): min 4 · mean 921 · max 4671
[VGGT-Omega @ 300] already failed (skipped_precautionary), skipping
